In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

project_path = "/content/drive/MyDrive/arabic_bert_project"

os.makedirs(project_path, exist_ok=True)

print("Project folder created!")

Project folder created!


In [ ]:
folders = ["data", "models", "notebooks"]

for folder in folders:
    os.makedirs(os.path.join(project_path, folder), exist_ok=True)

print("All folders ready!")

All folders ready!


In [ ]:
!pip install transformers
!pip install datasets
!pip install torch

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "aubmindlab/bert-base-arabertv2"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=5
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/611 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: aubmindlab/bert-base-arabertv2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider

In [ ]:
text = "هذا المشروع رائع جدا"

inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

outputs = model(**inputs)

print(outputs)

SequenceClassifierOutput(loss=None, logits=tensor([[ 1.1393, -0.1707, -0.5224, -0.7421, -0.3103]],
       grad_fn=<AddmmBackward0>), hidden_states=None, attentions=None)


In [ ]:
from datasets import load_dataset

dataset = load_dataset("labr")

dataset

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/3.83M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/919k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/11760 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2935 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 11760
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2935
    })
})

In [ ]:
dataset["train"][0]

{'text': ' هى رواية مملة لابعد الحدود مفيهاش اى حاجة تشد الا انها صغيرة مفيهاش جديد يعنى ولا فى القصة ولا فى الاسلوب ولا اى حاجة واحدة بترفض مغريات الرشوة رغم حاجتها للفلوس كذا مرة و تدى درس لزمايلها عن رفضها و انها هاتفضل نضيفة و بموقف ساذج اوى تقتنع و تقبل و مش بس تقبل لا بتتخلى عن مبادئها كلها :) بمبدأ خربانة خربانة مثلا فى اول الرواية قعد يذكر ان المصلحة كلها رجالة و مفيهاش اى ستات و دى اول دفعة من الستات تشتغل فيها حاجة ملهاش اى لازمة انها تتذكر اعتقد كمان الاسماء متلغبطة كل شوية اسم من الشخصيات يتغير ',
 'label': 0}

In [ ]:
def tokenize_function(example):
    return tokenizer(example["text"], padding="max_length", truncation=True)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/11760 [00:00<?, ? examples/s]

Map:   0%|          | 0/2935 [00:00<?, ? examples/s]

In [ ]:
print(dataset["train"].column_names)

['text', 'label']


In [ ]:
tokenized_datasets.set_format("torch")

In [ ]:
# ================================
# 1. تثبيت المكتبات
# ================================
!pip install transformers datasets torch -q

# ================================
# 2. ربط Google Drive
# ================================
from google.colab import drive
drive.mount('/content/drive')

# ================================
# 3. تحميل المكتبات
# ================================
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset
import torch

# ================================
# 4. تحميل الموديل (مع عدد الفئات)
# ================================
model_name = "aubmindlab/bert-base-arabertv2"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=5
)

# ================================
# 5. تحميل البيانات
# ================================
dataset = load_dataset("labr")

# ================================
# 6. Tokenization
# ================================
def tokenize_function(example):
    return tokenizer(example["text"], padding="max_length", truncation=True)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

# ================================
# 7. تجهيز البيانات
# ================================
tokenized_datasets = tokenized_datasets.remove_columns(["text"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")

# ================================
# 8. زيادة حجم البيانات (تحسين الدقة)
# ================================
small_train_dataset = tokenized_datasets["train"].shuffle(seed=42).select(range(2000))
small_test_dataset = tokenized_datasets["test"].shuffle(seed=42).select(range(400))

# ================================
# 9. إعداد التدريب (محسّن)
# ================================
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/arabic_bert_project/models",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=2,   # 👈 مهم للتحسين
    weight_decay=0.01,
    logging_dir="/content/logs",
)

# ================================
# 10. Trainer
# ================================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=small_test_dataset,
)

# ================================
# 11. التدريب
# ================================
trainer.train()

# ================================
# 12. التقييم
# ================================
trainer.evaluate()

# ================================
# 13. حفظ الموديل
# ================================
trainer.save_model("/content/drive/MyDrive/arabic_bert_project/models/my_model")

# ================================
# 14. تجربة النموذج
# ================================
model.eval()

text = "هذا المنتج ممتاز جدا وأنصح به"

inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits
predicted_class_id = torch.argmax(logits, dim=1).item()

labels_map = {
    0: "سيء جدا",
    1: "سيء",
    2: "محايد",
    3: "جيد",
    4: "ممتاز"
}

print("النص:", text)
print("التصنيف:", labels_map[predicted_class_id])

print("✅ تم التدريب والتجربة بنجاح!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: aubmindlab/bert-base-arabertv2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider

Map:   0%|          | 0/2935 [00:00<?, ? examples/s]

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Step,Training Loss
500,1.529422
1000,1.261664


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

RuntimeError: Expected all tensors to be on the same device, but got index is on cpu, different from other tensors on cuda:0 (when checking argument in method wrapper_CUDA__index_select)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)

text = "هذا المنتج ممتاز جدا وأنصح به"

inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

# 👇 ننقل البيانات إلى نفس الجهاز
inputs = {key: value.to(device) for key, value in inputs.items()}

model.eval()

with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits
predicted_class_id = torch.argmax(logits, dim=1).item()

labels_map = {
    0: "سيء جدا",
    1: "سيء",
    2: "محايد",
    3: "جيد",
    4: "ممتاز"
}

print("النص:", text)
print("التصنيف:", labels_map[predicted_class_id])

النص: هذا المنتج ممتاز جدا وأنصح به
التصنيف: ممتاز


In [ ]:
from transformers import pipeline

ner = pipeline(
    "ner",
    model="CAMeL-Lab/bert-base-arabic-camelbert-msa-ner",
    tokenizer="CAMeL-Lab/bert-base-arabic-camelbert-msa-ner"
)

text = "زار محمد بن سلمان مدينة الرياض أمس"

results = ner(text)

for entity in results:
    print(f"الكلمة: {entity['word']} | النوع: {entity['entity']}")

config.json:   0%|          | 0.00/980 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: CAMeL-Lab/bert-base-arabic-camelbert-msa-ner
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.pooler.dense.bias       | UNEXPECTED |  | 
bert.embeddings.position_ids | UNEXPECTED |  | 
bert.pooler.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/86.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

الكلمة: محمد | النوع: B-PERS
الكلمة: بن | النوع: I-PERS
الكلمة: سلمان | النوع: I-PERS
الكلمة: الرياض | النوع: B-LOC


In [ ]:
!pip install pyarabic -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.4/126.4 kB 3.5 MB/s eta 0:00:00


In [ ]:
!pip install sumy -q

from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.lsa import LsaSummarizer

text = """
الذكاء الاصطناعي أصبح من أهم التقنيات الحديثة التي تستخدم في العديد من المجالات مثل الطب والتعليم والصناعة.
يساعد الذكاء الاصطناعي في تحسين الكفاءة وتوفير الوقت والجهد، كما يساهم في اتخاذ قرارات دقيقة بناءً على البيانات.
كما أن استخدام الذكاء الاصطناعي يزداد بشكل كبير في السنوات الأخيرة.
"""

parser = PlaintextParser.from_string(text, Tokenizer("arabic"))
summarizer = LsaSummarizer()

summary = summarizer(parser.document, 2)  # عدد الجمل في الملخص

print("الملخص:")
for sentence in summary:
    print(sentence)

الملخص:
يساعد الذكاء الاصطناعي في تحسين الكفاءة وتوفير الوقت والجهد،
كما أن استخدام الذكاء الاصطناعي يزداد بشكل كبير في السنوات الأخيرة.


In [ ]:
!pip install --upgrade transformers

In [ ]:
def improved_qa(context, question):
    sentences = context.split(".")

    question_words = question.split()
    best_sentence = ""
    max_score = 0

    for sentence in sentences:
        score = sum(1 for word in question_words if word in sentence)

        if score > max_score:
            max_score = score
            best_sentence = sentence

    return best_sentence.strip()


context = """
الذكاء الاصطناعي يستخدم في مجالات متعددة مثل الطب والتعليم والصناعة.
يساعد في تحسين الكفاءة واتخاذ القرارات.
"""

question = "في أي مجالات يستخدم الذكاء الاصطناعي؟"

answer = improved_qa(context, question)

print("السؤال:", question)
print("الإجابة:", answer)

السؤال: في أي مجالات يستخدم الذكاء الاصطناعي؟
الإجابة: الذكاء الاصطناعي يستخدم في مجالات متعددة مثل الطب والتعليم والصناعة


In [ ]:
!pip install streamlit pyngrok -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 51.0 MB/s eta 0:00:00


In [ ]:
import ipywidgets as widgets
from IPython.display import display
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

# =========================
# تحميل الموديل (مرة واحدة فقط)
# =========================
model_path = "/content/drive/MyDrive/arabic_bert_project/models/my_model"

tokenizer = AutoTokenizer.from_pretrained("aubmindlab/bert-base-arabertv2")
model = AutoModelForSequenceClassification.from_pretrained(model_path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

labels_map = {
    0: "سيء جدا",
    1: "سيء",
    2: "محايد",
    3: "جيد",
    4: "ممتاز"
}

# =========================
# QA function
# =========================
def improved_qa(context, question):
    sentences = context.split(".")
    question_words = question.split()

    best_sentence = ""
    max_score = 0

    for sentence in sentences:
        score = sum(1 for word in question_words if word in sentence)
        if score > max_score:
            max_score = score
            best_sentence = sentence

    return best_sentence.strip()

# =========================
# واجهة المستخدم
# =========================
text_input = widgets.Textarea(
    value='',
    placeholder='اكتب النص هنا...',
    description='النص:',
    layout=widgets.Layout(width='100%', height='100px')
)

task_dropdown = widgets.Dropdown(
    options=['تحليل المشاعر', 'تلخيص النص', 'سؤال وجواب'],
    description='المهمة:'
)

question_input = widgets.Text(
    value='',
    placeholder='اكتب السؤال هنا...',
    description='السؤال:'
)

button = widgets.Button(description="تشغيل 🚀")
output = widgets.Output()

# =========================
# عند الضغط
# =========================
def on_button_click(b):
    with output:
        output.clear_output()
        text = text_input.value

        if not text.strip():
            print("⚠️ أدخل نص أولاً")
            return

        task = task_dropdown.value

        if task == "تحليل المشاعر":
            inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
            inputs = {k: v.to(device) for k, v in inputs.items()}

            with torch.no_grad():
                outputs = model(**inputs)

            pred = torch.argmax(outputs.logits, dim=1).item()
            print("💡 النتيجة:", labels_map[pred])

        elif task == "تلخيص النص":
            sentences = text.split(".")
            summary = ".".join(sentences[:2])
            print("✂️ الملخص:")
            print(summary)

        elif task == "سؤال وجواب":
            question = question_input.value
            if not question.strip():
                print("⚠️ أدخل السؤال")
                return

            answer = improved_qa(text, question)
            print("💬 الإجابة:")
            print(answer)

button.on_click(on_button_click)

# =========================
# عرض الواجهة
# =========================
display(text_input, task_dropdown, question_input, button, output)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Textarea(value='', description='النص:', layout=Layout(height='100px', width='100%'), placeholder='اكتب النص هن…

Dropdown(description='المهمة:', options=('تحليل المشاعر', 'تلخيص النص', 'سؤال وجواب'), value='تحليل المشاعر')

Text(value='', description='السؤال:', placeholder='اكتب السؤال هنا...')

Button(description='تشغيل 🚀', style=ButtonStyle())

Output()

In [ ]:
!streamlit run app.py &>/dev/null &

import time
time.sleep(5)

from google.colab.output import eval_js

print(eval_js("google.colab.kernel.proxyPort(8501)"))

https://8501-gpu-t4-s-kkb-usw1b2-22g0nz99esyut-b.us-west1-2.prod.colab.dev


In [ ]:
import os

base = "/content/drive/MyDrive/arabic_nlp_project"

folders = ["data", "models", "notebooks", "app"]

for f in folders:
    os.makedirs(os.path.join(base, f), exist_ok=True)

print("تم إنشاء الهيكل")

تم إنشاء الهيكل


In [ ]:
import shutil

src = "/content/drive/MyDrive/arabic_bert_project/models/my_model"
dst = "/content/drive/MyDrive/arabic_nlp_project/models/my_model"

shutil.copytree(src, dst, dirs_exist_ok=True)

print("تم نقل الموديل")

تم نقل الموديل


In [ ]:
%%writefile /content/drive/MyDrive/arabic_nlp_project/app/app.py

Writing /content/drive/MyDrive/arabic_nlp_project/app/app.py


In [ ]:
%%writefile /content/drive/MyDrive/arabic_nlp_project/app/app.py

print("واجهة المشروع")

Overwriting /content/drive/MyDrive/arabic_nlp_project/app/app.py
